# Data Preparation

Final raw-to-beat-signal dataset builder for the MI localization experiments.

## 1. Imports and Config

In [ ]:
# Data Prep: build strict 4-class beat-signal datasets directly from raw PTB-XL and PTB Diagnostic ECG.
import ast
import json
import os
import re
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import wfdb
from IPython.display import display
from scipy.signal import butter, decimate, filtfilt, find_peaks
from sklearn.model_selection import StratifiedShuffleSplit
from tqdm.auto import tqdm

warnings.filterwarnings('ignore', category=UserWarning, module='wfdb')
warnings.filterwarnings('ignore', message='.*invalid value encountered.*')

CURRENT_DIR = Path.cwd().resolve()
PROJECT_ROOT = CURRENT_DIR.parent if CURRENT_DIR.name == 'notebook' else CURRENT_DIR
RAW_SEARCH_ROOT = PROJECT_ROOT.parent


def env_path(name, default=None):
    value = os.environ.get(name, '').strip()
    if value in {'', '0'}:
        return default
    return value


def env_int(name, default):
    value = os.environ.get(name, '').strip()
    if value in {'', '0'}:
        return default
    try:
        parsed = int(value)
    except ValueError:
        return default
    return parsed if parsed > 0 else default

RAW_PATH_CONFIG = {
    'ptb_xl': env_path('CLICNET_PTB_XL_RAW_ROOT'),
    'ptb_diagnostic': env_path('CLICNET_PTB_DIAGNOSTIC_RAW_ROOT'),
}
OUTPUT_ROOT = PROJECT_ROOT / 'dataset'
FIGURE_ROOT = PROJECT_ROOT / 'outputs' / '1_fig_data_prep' / 'beat_examples'

DATASETS = ['ptb_xl', 'ptb_diagnostic']
SPLITS = ['train', 'val', 'test']
MAIN_LABELS = ['Normal', 'Anterior', 'Inferior', 'Lateral']
MAIN_LABEL_TO_INDEX = {name: idx for idx, name in enumerate(MAIN_LABELS)}
STRICT_SUB_TO_MAIN = {
    'normal': 'Normal',
    'anterior': 'Anterior',
    'inferior': 'Inferior',
    'lateral': 'Lateral',
    'antero-lateral': 'Lateral',
    'infero-lateral': 'Lateral',
    'postero-lateral': 'Lateral',
    'infero-postero-lateral': 'Lateral',
}

SOURCE_LEAD_ORDER = ['I', 'II', 'III', 'aVR', 'aVL', 'aVF', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6']
ALPA_LEAD_ORDER = ['I', 'aVL', 'II', 'III', 'aVF', 'aVR', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6']
SOURCE_TO_ALPA_IDX = [SOURCE_LEAD_ORDER.index(lead) for lead in ALPA_LEAD_ORDER]
LEAD_PRIOR_BY_MAIN = {
    'Normal': ALPA_LEAD_ORDER,
    'Anterior': ['V1', 'V2', 'V3', 'V4'],
    'Inferior': ['II', 'III', 'aVF'],
    'Lateral': ['I', 'aVL', 'V5', 'V6'],
}

TARGET_FS = 100
PTB_DIAGNOSTIC_RAW_FS = 1000
PTB_DIAGNOSTIC_STANDARD_SECONDS = 10
PTB_DIAGNOSTIC_STANDARD_SAMPLES = PTB_DIAGNOSTIC_STANDARD_SECONDS * TARGET_FS
PTB_DIAGNOSTIC_LATERAL_SECONDS = 60
PTB_DIAGNOSTIC_LATERAL_SAMPLES = PTB_DIAGNOSTIC_LATERAL_SECONDS * TARGET_FS
PTB_DIAGNOSTIC_FULL_LENGTH_LABELS = set()
PTB_DIAGNOSTIC_ALLOWED_LATERAL_LABELS = {'lateral', 'antero-lateral', 'infero-lateral', 'postero-lateral', 'infero-postero-lateral'}
PTB_DIAGNOSTIC_STRICT_LATERAL_TEST_LABEL = 'lateral'
PTB_DIAGNOSTIC_LATERAL_TRAIN_RECORDS = 50
PTB_DIAGNOSTIC_LATERAL_VAL_RECORDS = 8
PTB_DIAGNOSTIC_LATERAL_TEST_RECORDS = 1
PTB_DIAGNOSTIC_FIXED_LATERAL_TEST_RECORD_ID = 's0141lre'
PTB_DIAGNOSTIC_FIXED_LATERAL_TEST_PATIENT_ID = 'patient043'
PTB_XL_LATERAL_INVOLVED_CODES = {'LMI', 'ALMI', 'ILMI', 'IPLMI'}
PTB_DIAGNOSTIC_FORCE_TEST_PATIENTS_BY_LABEL = {}
PTB_DIAGNOSTIC_FORCE_TRAIN_PATIENTS_BY_LABEL = {}
PTB_DIAGNOSTIC_DROP_BOUNDARY_BEATS = 1
BASELINE_CUTOFF = 0.5
BASELINE_ORDER = 2
BANDPASS_LOWCUT = 0.5
BANDPASS_HIGHCUT = 40.0
BANDPASS_ORDER = 4

BEAT_FS = 100
RPEAK_LEAD_NAME = 'II'
RPEAK_LEAD_INDEX = ALPA_LEAD_ORDER.index(RPEAK_LEAD_NAME)
PRE_BEAT_SAMPLES = int(0.20 * BEAT_FS)
POST_BEAT_SAMPLES = int(0.45 * BEAT_FS)
BEAT_LENGTH = PRE_BEAT_SAMPLES + POST_BEAT_SAMPLES
RECORD_TEST_SIZE = 0.20
RECORD_VAL_SIZE_FROM_REMAINING = 0.125
RANDOM_SEED = env_int('CLICNET_SEED', 42)

for path in [OUTPUT_ROOT, FIGURE_ROOT]:
    path.mkdir(parents=True, exist_ok=True)

print(f'Project root: {PROJECT_ROOT}')
print(f'Output root: {OUTPUT_ROOT}')


## 2. Label Mapping

In [ ]:
def find_raw_root(required_file, preferred_names, explicit_path=None):
    if explicit_path:
        explicit = Path(explicit_path).expanduser().resolve()
        if not (explicit / required_file).exists():
            raise FileNotFoundError(f'Configured raw path does not contain {required_file}: {explicit}')
        return explicit

    candidates = []
    search_bases = [
        PROJECT_ROOT,
        *PROJECT_ROOT.parents,
        Path.home() / 'code-program' / 'code-thesis',
        Path.home() / 'code-program' / 'code-thesis' / 'hibah',
    ]
    seen_bases = []
    for base in search_bases:
        if base.exists() and base not in seen_bases:
            seen_bases.append(base)
            for current_root, dirs, files in os.walk(base):
                dirs[:] = [d for d in dirs if d not in {'dump_artifact', '.git', '.venv', 'outputs'}]
                if required_file in files:
                    candidates.append(Path(current_root))
    if not candidates:
        searched = ', '.join(str(base) for base in seen_bases)
        raise FileNotFoundError(f'Cannot find raw file {required_file}. Searched: {searched}. Set RAW_PATH_CONFIG manually.')
    for preferred in preferred_names:
        for candidate in candidates:
            if preferred in str(candidate):
                return candidate
    return sorted(candidates, key=lambda p: len(str(p)))[0]

PTB_XL_ROOT = find_raw_root('ptbxl_database.csv', ['ptb-xl', 'ptbxl'], RAW_PATH_CONFIG.get('ptb_xl'))
PTB_DIAGNOSTIC_ROOT = find_raw_root('data_raw.npz', ['ptb-diagnostic', 'diagnostic'], RAW_PATH_CONFIG.get('ptb_diagnostic'))
print(f'PTB-XL raw root: {PTB_XL_ROOT}')
print(f'PTB Diagnostic raw root: {PTB_DIAGNOSTIC_ROOT}')


def normalize_label_text(value):
    if value is None or pd.isna(value):
        return ''
    text = str(value).strip().lower()
    text = text.replace('_', '-').replace('/', '-').replace(',', ' ')
    text = re.sub(r'\([^)]*\)', ' ', text)
    text = re.sub(r'[^a-z0-9\- ]+', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    aliases = {
        'healty control': 'healthy control',
        'healthy-control': 'healthy control',
        'antero': 'anterior',
        'infero': 'inferior',
        'anteroseptal': 'antero-septal',
        'anterolateral': 'antero-lateral',
        'inferolateral': 'infero-lateral',
        'infero-latera': 'infero-lateral',
        'inferoposterior': 'infero-posterior',
        'posterolateral': 'postero-lateral',
        'inferoposterolateral': 'infero-postero-lateral',
    }
    return aliases.get(text, text)


def main_vector_from_label(main_label):
    vector = np.zeros(4, dtype=np.float32)
    vector[MAIN_LABEL_TO_INDEX[main_label]] = 1.0
    return vector


def territory_vector_from_label(main_label):
    vector = np.zeros(3, dtype=np.float32)
    if main_label == 'Anterior':
        vector[0] = 1.0
    elif main_label == 'Inferior':
        vector[1] = 1.0
    elif main_label == 'Lateral':
        vector[2] = 1.0
    return vector


def lead_prior_from_label(main_label):
    weights = np.zeros(len(ALPA_LEAD_ORDER), dtype=np.float32)
    for lead in LEAD_PRIOR_BY_MAIN[main_label]:
        weights[ALPA_LEAD_ORDER.index(lead)] = 1.0
    return weights / weights.sum()


def vector_json(vector):
    return json.dumps([float(x) for x in np.asarray(vector, dtype=np.float32)])


def ptbxl_strict_label(row, scp_diag):
    codes = row['scp_codes'] if isinstance(row['scp_codes'], dict) else ast.literal_eval(row['scp_codes'])
    code_set = set(codes.keys()) if isinstance(codes, dict) else set(codes)
    if code_set & PTB_XL_LATERAL_INVOLVED_CODES:
        return 'lateral'
    subclasses = []
    for code in code_set:
        if code in scp_diag.index:
            subclass = str(scp_diag.loc[code, 'diagnostic_subclass'])
            subclasses.append(subclass)
    subclass_set = set(subclasses)
    strict_map = {'NORM': 'normal', 'AMI': 'anterior', 'IMI': 'inferior'}
    strict_found = sorted({strict_map[item] for item in subclass_set if item in strict_map})
    if len(strict_found) != 1:
        return None
    non_strict_mi = subclass_set - set(strict_map.keys())
    if non_strict_mi:
        return None
    return strict_found[0]


def ptb_diagnostic_strict_label(row):
    reason = normalize_label_text(row.get('Reason_for_admission', ''))
    acute = normalize_label_text(row.get('Acute_infarction_(localization)', ''))
    former = normalize_label_text(row.get('Former_infarction_(localization)', ''))
    empty = {'', 'no', 'unknown', 'nan', 'none'}
    if reason in {'healthy control', 'normal'} and acute in empty and former in empty:
        return 'normal'

    candidates = [normalize_label_text(item) for item in [acute, former] if normalize_label_text(item) not in empty]
    lateral_candidates = [item for item in candidates if item in PTB_DIAGNOSTIC_ALLOWED_LATERAL_LABELS]
    if lateral_candidates:
        return lateral_candidates[0]

    strict_candidates = [item for item in candidates if item in {'anterior', 'inferior'}]
    if len(set(strict_candidates)) == 1 and len(strict_candidates) == len(candidates):
        return strict_candidates[0]
    return None


def make_label_row(dataset_source, record_id, patient_id, raw_label, clinical_sub_label, signal_path, source_index):
    main_label = STRICT_SUB_TO_MAIN[clinical_sub_label]
    main_vector = main_vector_from_label(main_label)
    territory_vector = territory_vector_from_label(main_label)
    lead_prior = lead_prior_from_label(main_label)
    return {
        'record_id': str(record_id),
        'dataset_source': dataset_source,
        'source_index': int(source_index) if isinstance(source_index, (int, np.integer)) else str(source_index),
        'patient_id': str(patient_id),
        'raw_label': str(raw_label),
        'clinical_sub_label': clinical_sub_label,
        'main_label_name': main_label,
        'main_label_vector': vector_json(main_vector),
        'territory_vector': vector_json(territory_vector),
        'lead_prior_vector': vector_json(lead_prior),
        'signal_path': str(signal_path),
        'fs': TARGET_FS,
        'lead_order': json.dumps(ALPA_LEAD_ORDER),
    }


def parse_vector(value, dtype=np.float32):
    if isinstance(value, str):
        return np.asarray(json.loads(value), dtype=dtype)
    return np.asarray(value, dtype=dtype)


## 3. Signal Processing and Beat Extraction

In [ ]:
def normalize_source_lead_name(name):
    text = str(name).strip()
    lookup = {lead.upper(): lead for lead in SOURCE_LEAD_ORDER}
    return lookup.get(text.upper(), text)


def validate_source_lead_order(sig_names, record_id='record'):
    actual = [normalize_source_lead_name(name) for name in list(sig_names)[:len(SOURCE_LEAD_ORDER)]]
    if actual != SOURCE_LEAD_ORDER:
        raise ValueError(f'{record_id}: expected source lead order {SOURCE_LEAD_ORDER}, got {actual}')


def reorder_to_alpa(signal):
    return np.asarray(signal[:, SOURCE_TO_ALPA_IDX], dtype=np.float32)


def butterworth_bandpass_filter(data, fs, lowcut=BANDPASS_LOWCUT, highcut=BANDPASS_HIGHCUT, order=BANDPASS_ORDER, axis=0):
    nyquist = 0.5 * fs
    b, a = butter(order, [lowcut / nyquist, highcut / nyquist], btype='band')
    return filtfilt(b, a, data, axis=axis)


def reduce_baseline_wander(data, fs, cutoff=BASELINE_CUTOFF, order=BASELINE_ORDER, axis=0):
    nyquist = 0.5 * fs
    b, a = butter(order, cutoff / nyquist, btype='low')
    baseline = filtfilt(b, a, data, axis=axis)
    return data - baseline


def preprocess_signal(signal, fs=TARGET_FS):
    signal = np.nan_to_num(np.asarray(signal, dtype=np.float32), nan=0.0, posinf=0.0, neginf=0.0)
    filtered = reduce_baseline_wander(signal, fs=fs, axis=0)
    filtered = butterworth_bandpass_filter(filtered, fs=fs, axis=0)
    return filtered.astype(np.float32)


def downsample_ptb_diagnostic(signal, source_fs=PTB_DIAGNOSTIC_RAW_FS, target_fs=TARGET_FS):
    if source_fs == target_fs:
        return signal.astype(np.float32)
    factor = int(source_fs // target_fs)
    if source_fs % target_fs != 0:
        raise ValueError(f'Expected integer downsample factor from {source_fs} to {target_fs}')
    return decimate(signal, factor, axis=0, zero_phase=True).astype(np.float32)


def detect_r_peaks_for_beats(ecg, fs=BEAT_FS):
    signal_abs = np.abs(np.asarray(ecg, dtype=np.float32))
    if signal_abs.size == 0 or np.nanmax(signal_abs) <= 0:
        return np.asarray([], dtype=np.int64)
    distance = max(1, int(0.30 * fs))
    candidate_peaks, _ = find_peaks(signal_abs, distance=distance)
    if len(candidate_peaks) == 0:
        return np.asarray([], dtype=np.int64)
    r_peaks = []
    rr_intervals = []
    spki = float(np.nanmax(signal_abs) * 0.5)
    npki = float(np.nanmean(signal_abs) * 0.5)
    recent_rr = []
    for peak in candidate_peaks:
        peak_val = float(signal_abs[peak])
        threshold_1 = npki + 0.25 * (spki - npki)
        threshold_2 = 0.5 * threshold_1
        if peak_val > threshold_1:
            if len(r_peaks) == 0 or (peak - r_peaks[-1]) > 0.36 * fs:
                r_peaks.append(int(peak))
                spki = 0.125 * peak_val + 0.875 * spki
                if len(r_peaks) >= 2:
                    rr_intervals.append(r_peaks[-1] - r_peaks[-2])
                    recent_rr = rr_intervals[-8:]
        else:
            npki = 0.125 * peak_val + 0.875 * npki
        if len(r_peaks) > 0 and recent_rr:
            last_rr = float(np.mean(recent_rr))
            if (peak - r_peaks[-1]) > 1.5 * last_rr and peak_val > threshold_2 and (peak - r_peaks[-1]) > 0.36 * fs:
                r_peaks.append(int(peak))
                spki = 0.25 * peak_val + 0.75 * spki
                if len(r_peaks) >= 2:
                    rr_intervals.append(r_peaks[-1] - r_peaks[-2])
                    recent_rr = rr_intervals[-8:]
    return np.asarray(sorted(set(r_peaks)), dtype=np.int64)


def build_record_beats(record_signal, record_row):
    r_peaks = detect_r_peaks_for_beats(record_signal[:, RPEAK_LEAD_INDEX], fs=BEAT_FS)
    if str(record_row.get('dataset_source', '')) == 'ptb_diagnostic' and PTB_DIAGNOSTIC_DROP_BOUNDARY_BEATS > 0:
        drop_n = int(PTB_DIAGNOSTIC_DROP_BOUNDARY_BEATS)
        if len(r_peaks) > 2 * drop_n:
            r_peaks = r_peaks[drop_n:-drop_n]
        else:
            r_peaks = np.asarray([], dtype=np.int64)
    rows, arrays = [], []
    for beat_index, r_peak in enumerate(r_peaks):
        start = int(r_peak) - PRE_BEAT_SAMPLES
        end = int(r_peak) + POST_BEAT_SAMPLES
        if start < 0 or end > record_signal.shape[0]:
            continue
        beat_signal = np.asarray(record_signal[start:end, :], dtype=np.float32)
        if beat_signal.shape != (BEAT_LENGTH, len(ALPA_LEAD_ORDER)):
            continue
        row = dict(record_row)
        row.update({
            'beat_id': f"{record_row['record_id']}_beat{beat_index}",
            'beat_index': int(beat_index),
            'r_peak_sample': int(r_peak),
            'start_sample': int(start),
            'end_sample': int(end),
            'fs': BEAT_FS,
            'rpeak_lead': RPEAK_LEAD_NAME,
        })
        rows.append(row)
        arrays.append(beat_signal)
    return rows, arrays


## 4. Raw Data Loading

In [ ]:
def load_ptbxl_records():
    meta = pd.read_csv(PTB_XL_ROOT / 'ptbxl_database.csv', index_col='ecg_id')
    meta['scp_codes'] = meta['scp_codes'].apply(ast.literal_eval)
    scp_df = pd.read_csv(PTB_XL_ROOT / 'scp_statements.csv', index_col=0)
    scp_diag = scp_df[scp_df['diagnostic'] == 1]

    rows, signals, skipped = [], [], []
    for source_index, (ecg_id, row) in enumerate(tqdm(meta.iterrows(), total=len(meta), desc='PTB-XL raw records')):
        strict_label = ptbxl_strict_label(row, scp_diag)
        if strict_label is None:
            skipped.append({'record_id': ecg_id, 'reason': 'not_strict_single_main_label'})
            continue
        filename = row['filename_lr']
        signal, header = wfdb.rdsamp(str(PTB_XL_ROOT / filename))
        validate_source_lead_order(header.get('sig_name', []), record_id=ecg_id)
        signal = reorder_to_alpa(signal[:, :len(SOURCE_LEAD_ORDER)])
        signal = preprocess_signal(signal, fs=TARGET_FS)
        raw_label = '|'.join(row['scp_codes'].keys())
        label_row = make_label_row('ptb_xl', ecg_id, row.get('patient_id', ecg_id), raw_label, strict_label, filename, source_index)
        rows.append(label_row)
        signals.append(signal)
    labels = pd.DataFrame(rows)
    skipped_df = pd.DataFrame(skipped)
    return labels, signals, skipped_df


def load_ptb_diagnostic_records():
    meta = pd.read_csv(PTB_DIAGNOSTIC_ROOT / 'meta.csv')
    raw_npz = np.load(PTB_DIAGNOSTIC_ROOT / 'data_raw.npz')
    rows, signals, skipped = [], [], []
    for source_index, row in tqdm(meta.iterrows(), total=len(meta), desc='PTB Diagnostic raw records'):
        strict_label = ptb_diagnostic_strict_label(row)
        record_id = str(row.get('record_id', source_index))
        patient = str(row.get('patient', record_id))
        key = f'{patient}/{record_id}'
        if strict_label is None:
            skipped.append({'record_id': record_id, 'patient_id': patient, 'reason': 'not_strict_single_main_label'})
            continue
        if key not in raw_npz:
            skipped.append({'record_id': record_id, 'patient_id': patient, 'reason': f'missing_npz_key:{key}'})
            continue
        signal = np.asarray(raw_npz[key], dtype=np.float32)[:, :len(SOURCE_LEAD_ORDER)]
        signal = reorder_to_alpa(signal)
        signal = downsample_ptb_diagnostic(signal, source_fs=PTB_DIAGNOSTIC_RAW_FS, target_fs=TARGET_FS)
        signal = preprocess_signal(signal, fs=TARGET_FS)
        use_fixed_lateral_test_length = (
            record_id == PTB_DIAGNOSTIC_FIXED_LATERAL_TEST_RECORD_ID
            and patient == PTB_DIAGNOSTIC_FIXED_LATERAL_TEST_PATIENT_ID
            and strict_label == PTB_DIAGNOSTIC_STRICT_LATERAL_TEST_LABEL
        )
        required_samples = PTB_DIAGNOSTIC_LATERAL_SAMPLES if use_fixed_lateral_test_length else PTB_DIAGNOSTIC_STANDARD_SAMPLES
        signal_policy = 'fixed_lateral_test_first_60_seconds' if use_fixed_lateral_test_length else 'first_10_seconds'
        if signal.shape[0] < required_samples:
            skipped.append({'record_id': record_id, 'patient_id': patient, 'reason': f'shorter_than_required_seconds_after_downsample:{required_samples / TARGET_FS:.0f}s'})
            continue
        signal = signal[:required_samples, :]
        raw_label = '|'.join(str(row.get(col, '')) for col in ['Reason_for_admission', 'Acute_infarction_(localization)', 'Former_infarction_(localization)'])
        label_row = make_label_row('ptb_diagnostic', record_id, patient, raw_label, strict_label, key, source_index)
        label_row.update({
            'source_signal_samples': int(signal.shape[0]),
            'source_signal_seconds': float(signal.shape[0] / TARGET_FS),
            'signal_length_policy': signal_policy,
        })
        rows.append(label_row)
        signals.append(signal)
    labels = pd.DataFrame(rows)
    skipped_df = pd.DataFrame(skipped)
    return labels, signals, skipped_df


## 5. Split and Export Beat Signals

In [ ]:
def split_indices_with_stratification(df, label_col, test_size, seed):
    df = df.reset_index(drop=False).rename(columns={'index': '_source_index'})
    indices = np.arange(len(df))
    y = df[label_col].astype(str).to_numpy()
    if len(df) < 2 or pd.Series(y).nunique() < 2:
        n_test = max(1, int(round(len(df) * test_size))) if len(df) else 0
        ordered = df.sort_values('_source_index')
        test_local = ordered.tail(n_test).index.to_numpy(dtype=int)
        train_local = np.asarray([idx for idx in indices if idx not in set(test_local)], dtype=int)
    else:
        splitter = StratifiedShuffleSplit(n_splits=1, test_size=test_size, random_state=seed)
        train_local, test_local = next(splitter.split(indices, y))
    return df.iloc[train_local]['_source_index'].to_numpy(dtype=int), df.iloc[test_local]['_source_index'].to_numpy(dtype=int)


def make_record_wise_splits(labels_df, seed=RANDOM_SEED):
    labels_df = labels_df.reset_index(drop=True).copy()
    y = labels_df['main_label_name'].astype(str).to_numpy()
    indices = np.arange(len(labels_df))
    splitter = StratifiedShuffleSplit(n_splits=1, test_size=RECORD_TEST_SIZE, random_state=seed)
    train_val_idx, test_idx = next(splitter.split(indices, y))
    val_splitter = StratifiedShuffleSplit(n_splits=1, test_size=RECORD_VAL_SIZE_FROM_REMAINING, random_state=seed + 1)
    train_rel, val_rel = next(val_splitter.split(train_val_idx, y[train_val_idx]))
    train_idx = train_val_idx[train_rel]
    val_idx = train_val_idx[val_rel]
    splits = {'train': np.asarray(train_idx), 'val': np.asarray(val_idx), 'test': np.asarray(test_idx)}
    record_sets = {name: set(labels_df.iloc[idx]['record_id'].astype(str)) for name, idx in splits.items()}
    for i, left in enumerate(SPLITS):
        for right in SPLITS[i + 1:]:
            overlap = record_sets[left] & record_sets[right]
            assert not overlap, f'Record leakage between {left} and {right}: {list(overlap)[:5]}'
    return splits


def make_ptb_diagnostic_lateral_balanced_record_splits(labels_df, seed=RANDOM_SEED):
    labels_df = labels_df.reset_index(drop=True).copy()
    non_lateral_df = labels_df[labels_df['main_label_name'] != 'Lateral'].copy()
    lateral_df = labels_df[labels_df['main_label_name'] == 'Lateral'].copy()

    splits = {name: [] for name in SPLITS}
    if len(non_lateral_df):
        non_lateral_splits = make_record_wise_splits(non_lateral_df, seed=seed)
        for split_name, local_idx in non_lateral_splits.items():
            splits[split_name].extend(non_lateral_df.iloc[local_idx].index.to_list())

    if len(lateral_df):
        strict_lateral = lateral_df[lateral_df['clinical_sub_label'] == PTB_DIAGNOSTIC_STRICT_LATERAL_TEST_LABEL].copy()
        involved_train_pool = lateral_df[lateral_df['clinical_sub_label'] != PTB_DIAGNOSTIC_STRICT_LATERAL_TEST_LABEL].copy()

        strict_lateral = strict_lateral.sort_values(['patient_id', 'record_id']).reset_index().rename(columns={'index': '_source_index'})
        fixed_test = strict_lateral[
            (strict_lateral['record_id'].astype(str) == PTB_DIAGNOSTIC_FIXED_LATERAL_TEST_RECORD_ID)
            & (strict_lateral['patient_id'].astype(str) == PTB_DIAGNOSTIC_FIXED_LATERAL_TEST_PATIENT_ID)
        ]
        if fixed_test.empty:
            raise ValueError(
                f'Fixed lateral test record not found: '
                f'{PTB_DIAGNOSTIC_FIXED_LATERAL_TEST_PATIENT_ID}/{PTB_DIAGNOSTIC_FIXED_LATERAL_TEST_RECORD_ID}'
            )
        test_idx = fixed_test['_source_index'].head(PTB_DIAGNOSTIC_LATERAL_TEST_RECORDS).to_list()

        involved_train_pool = involved_train_pool.sample(frac=1.0, random_state=seed).reset_index().rename(columns={'index': '_source_index'})
        n_val = min(PTB_DIAGNOSTIC_LATERAL_VAL_RECORDS, len(involved_train_pool))
        val_idx = involved_train_pool.iloc[:n_val]['_source_index'].to_list()
        remaining = involved_train_pool.iloc[n_val:]
        n_train = min(PTB_DIAGNOSTIC_LATERAL_TRAIN_RECORDS, len(remaining))
        train_idx = remaining.iloc[:n_train]['_source_index'].to_list()

        splits['train'].extend(train_idx)
        splits['val'].extend(val_idx)
        splits['test'].extend(test_idx)

    splits = {name: np.asarray(sorted(set(values)), dtype=int) for name, values in splits.items()}
    record_sets = {name: set(labels_df.iloc[idx]['record_id'].astype(str)) for name, idx in splits.items()}
    for i, left in enumerate(SPLITS):
        for right in SPLITS[i + 1:]:
            overlap = record_sets[left] & record_sets[right]
            assert not overlap, f'PTB Diagnostic record leakage between {left} and {right}: {list(overlap)[:5]}'
    return splits


def make_ptb_diagnostic_patient_wise_splits(labels_df, seed=RANDOM_SEED):
    labels_df = labels_df.reset_index(drop=True).copy()
    patient_labels = (labels_df.groupby('patient_id', as_index=False)['main_label_name']
                      .agg(lambda values: values.mode().iat[0]))
    train_val_pat_idx, test_pat_idx = split_indices_with_stratification(patient_labels, 'main_label_name', RECORD_TEST_SIZE, seed)
    train_val_patients = patient_labels.iloc[train_val_pat_idx].reset_index(drop=True)
    train_pat_idx, val_pat_idx = split_indices_with_stratification(train_val_patients, 'main_label_name', RECORD_VAL_SIZE_FROM_REMAINING, seed + 1)
    patient_sets = {
        'train': set(train_val_patients.iloc[train_pat_idx]['patient_id'].astype(str)),
        'val': set(train_val_patients.iloc[val_pat_idx]['patient_id'].astype(str)),
        'test': set(patient_labels.iloc[test_pat_idx]['patient_id'].astype(str)),
    }
    for label_name, forced_patients in PTB_DIAGNOSTIC_FORCE_TRAIN_PATIENTS_BY_LABEL.items():
        available_for_label = set(patient_labels.loc[patient_labels['main_label_name'].eq(label_name), 'patient_id'].astype(str))
        forced = set(forced_patients) & available_for_label
        if forced:
            patient_sets['val'] -= forced
            patient_sets['test'] -= forced
            patient_sets['train'] |= forced
    for label_name, forced_patients in PTB_DIAGNOSTIC_FORCE_TEST_PATIENTS_BY_LABEL.items():
        available_for_label = set(patient_labels.loc[patient_labels['main_label_name'].eq(label_name), 'patient_id'].astype(str))
        forced = set(forced_patients) & available_for_label
        if forced:
            patient_sets['train'] -= forced
            patient_sets['val'] -= forced
            patient_sets['test'] |= forced
    splits = {}
    for split_name, patient_set in patient_sets.items():
        splits[split_name] = labels_df.index[labels_df['patient_id'].astype(str).isin(patient_set)].to_numpy(dtype=int)

    record_sets = {name: set(labels_df.iloc[idx]['record_id'].astype(str)) for name, idx in splits.items()}
    patient_sets_by_split = {name: set(labels_df.iloc[idx]['patient_id'].astype(str)) for name, idx in splits.items()}
    for i, left in enumerate(SPLITS):
        for right in SPLITS[i + 1:]:
            record_overlap = record_sets[left] & record_sets[right]
            patient_overlap = patient_sets_by_split[left] & patient_sets_by_split[right]
            assert not record_overlap, f'PTB Diagnostic record leakage between {left} and {right}: {list(record_overlap)[:5]}'
            assert not patient_overlap, f'PTB Diagnostic patient leakage between {left} and {right}: {list(patient_overlap)[:5]}'
    return splits

def collect_beats_for_records(dataset_name, labels_df, signals, record_indices, split_name, skipped_rows):
    beat_rows, beat_arrays = [], []
    for record_index in record_indices:
        record_row = labels_df.iloc[int(record_index)].to_dict()
        rows, arrays = build_record_beats(signals[int(record_index)], record_row)
        if not rows:
            skipped_rows.append({
                'dataset_source': dataset_name,
                'split': split_name,
                'record_id': record_row.get('record_id'),
                'patient_id': record_row.get('patient_id'),
                'main_label_name': record_row.get('main_label_name'),
                'clinical_sub_label': record_row.get('clinical_sub_label'),
                'reason': 'no_valid_beat_after_r_peak_detection',
            })
            continue
        beat_rows.extend(rows)
        beat_arrays.extend(arrays)
    return beat_rows, beat_arrays


def save_beat_split(dataset_name, split_name, beat_rows, beat_arrays):
    split_dir = OUTPUT_ROOT / dataset_name / split_name
    split_dir.mkdir(parents=True, exist_ok=True)
    meta = pd.DataFrame(beat_rows)
    x_beats = np.stack(beat_arrays).astype(np.float32) if beat_arrays else np.empty((0, BEAT_LENGTH, len(ALPA_LEAD_ORDER)), dtype=np.float32)
    main = np.vstack(meta['main_label_vector'].map(parse_vector).to_numpy()).astype(np.float32) if len(meta) else np.empty((0, 4), dtype=np.float32)
    territory = np.vstack(meta['territory_vector'].map(parse_vector).to_numpy()).astype(np.float32) if len(meta) else np.empty((0, 3), dtype=np.float32)
    prior = np.vstack(meta['lead_prior_vector'].map(parse_vector).to_numpy()).astype(np.float32) if len(meta) else np.empty((0, 12), dtype=np.float32)
    np.save(split_dir / 'x_beats.npy', x_beats)
    np.save(split_dir / 'main_label.npy', main)
    np.save(split_dir / 'territory.npy', territory)
    np.save(split_dir / 'lead_prior.npy', prior)
    meta.to_csv(split_dir / 'metadata.csv', index=False)
    return {
        'dataset_source': dataset_name,
        'split': split_name,
        'records': int(meta['record_id'].nunique()) if len(meta) else 0,
        'patients': int(meta['patient_id'].nunique()) if len(meta) and 'patient_id' in meta else 0,
        'beats': int(len(meta)),
        'signal_shape': str(tuple(x_beats.shape)),
    }


def plot_beat_examples(dataset_name, split_name, labels_df, x_beats, max_examples=4):
    if len(labels_df) == 0:
        return
    examples = labels_df.groupby('main_label_name', sort=False).head(1).head(max_examples)
    fig, axes = plt.subplots(len(examples), 1, figsize=(12, 3 * len(examples)), dpi=250, squeeze=False)
    for ax, (_, row) in zip(axes[:, 0], examples.iterrows()):
        beat_idx = int(row.name)
        signal = x_beats[beat_idx, :, RPEAK_LEAD_INDEX]
        ax.plot(np.arange(len(signal)) / BEAT_FS, signal, color='#d55e00', linewidth=1.8)
        ax.axvline(PRE_BEAT_SAMPLES / BEAT_FS, color='black', linestyle='--', linewidth=1.2)
        ax.set_title(f"{dataset_name} {split_name} | {row['main_label_name']} | {row['beat_id']}", fontsize=12)
        ax.set_xlabel('Time (s)')
        ax.set_ylabel(f'Lead {RPEAK_LEAD_NAME}')
        ax.grid(True, alpha=0.3)
    fig.tight_layout()
    fig.savefig(FIGURE_ROOT / f'{dataset_name}_{split_name}_beat_examples.png', bbox_inches='tight')
    plt.close(fig)


def export_dataset(dataset_name, labels_df, signals, source_skipped):
    dataset_dir = OUTPUT_ROOT / dataset_name
    dataset_dir.mkdir(parents=True, exist_ok=True)
    split_summary_rows = []
    skipped_rows = source_skipped.to_dict('records') if len(source_skipped) else []
    all_metadata = []

    if dataset_name == 'ptb_diagnostic':
        record_splits = make_ptb_diagnostic_lateral_balanced_record_splits(labels_df)
        split_policy = ('PTB Diagnostic record-wise split with balanced Lateral-involved sampling. '
                        'Normal/Anterior/Inferior use strict labels; Lateral train/val uses lateral-involved non-strict records and test uses strict lateral-only records. '
                        'All labels use the first 10 seconds before beat extraction. Boundary beats are dropped.')
    else:
        record_splits = make_record_wise_splits(labels_df)
        split_policy = 'Record-wise stratified split; no record appears in more than one split.'

    payloads = {}
    for split_name, record_indices in record_splits.items():
        beat_rows, beat_arrays = collect_beats_for_records(dataset_name, labels_df, signals, record_indices, split_name, skipped_rows)
        payloads[split_name] = (beat_rows, beat_arrays, record_indices)

    for split_name in SPLITS:
        beat_rows, beat_arrays, source_record_indices = payloads[split_name]
        for row in beat_rows:
            row['recordwise_split'] = split_name
        summary = save_beat_split(dataset_name, split_name, beat_rows, beat_arrays)
        summary['source_records_in_split'] = int(len(source_record_indices))
        split_meta = pd.DataFrame(beat_rows)
        for label in MAIN_LABELS:
            summary[f'{label}_beats'] = int((split_meta['main_label_name'] == label).sum()) if len(split_meta) else 0
            summary[f'{label}_records'] = int(split_meta.loc[split_meta['main_label_name'] == label, 'record_id'].nunique()) if len(split_meta) else 0
            summary[f'{label}_patients'] = int(split_meta.loc[split_meta['main_label_name'] == label, 'patient_id'].nunique()) if len(split_meta) and 'patient_id' in split_meta else 0
        split_summary_rows.append(summary)
        if len(split_meta):
            all_metadata.append(split_meta)
            plot_beat_examples(dataset_name, split_name, split_meta.reset_index(drop=True), np.load(OUTPUT_ROOT / dataset_name / split_name / 'x_beats.npy', mmap_mode='r'))

    split_summary = pd.DataFrame(split_summary_rows)
    split_summary.to_csv(dataset_dir / 'split_summary.csv', index=False)
    if all_metadata:
        pd.concat(all_metadata, ignore_index=True).to_csv(dataset_dir / 'metadata_all.csv', index=False)
    if skipped_rows:
        pd.DataFrame(skipped_rows).to_csv(dataset_dir / 'skipped_records.csv', index=False)

    with open(dataset_dir / 'config.json', 'w') as f:
        json.dump({
            'version': 'final_strict_4class_beat_signals_only_hybrid_ptb_diagnostic_split',
            'dataset_source': dataset_name,
            'sampling_rate_hz': BEAT_FS,
            'lead_order': ALPA_LEAD_ORDER,
            'source_lead_order': SOURCE_LEAD_ORDER,
            'main_label_order': MAIN_LABELS,
            'strict_sub_to_main': STRICT_SUB_TO_MAIN,
            'preprocessing': f'baseline removal {BASELINE_CUTOFF} Hz and Butterworth bandpass {BANDPASS_LOWCUT}-{BANDPASS_HIGHCUT} Hz',
            'beat_window': {'pre_samples': PRE_BEAT_SAMPLES, 'post_samples': POST_BEAT_SAMPLES, 'beat_length_samples': BEAT_LENGTH},
            'ptb_diagnostic_signal_policy': {
                'all_labels': 'first 10 seconds before beat extraction',
                'standard_seconds': PTB_DIAGNOSTIC_STANDARD_SECONDS,
                'lateral_label_policy': 'balanced: train uses up to 50 non-strict lateral-involved records, val uses up to 8 non-strict lateral-involved records, and test uses one fixed strict lateral-only record',
                'fixed_lateral_test_patient_id': PTB_DIAGNOSTIC_FIXED_LATERAL_TEST_PATIENT_ID,
                'fixed_lateral_test_record_id': PTB_DIAGNOSTIC_FIXED_LATERAL_TEST_RECORD_ID,
                'fixed_lateral_test_seconds': PTB_DIAGNOSTIC_LATERAL_SECONDS,
                'lateral_train_records_target': PTB_DIAGNOSTIC_LATERAL_TRAIN_RECORDS,
                'lateral_val_records_target': PTB_DIAGNOSTIC_LATERAL_VAL_RECORDS,
                'lateral_test_records_target': PTB_DIAGNOSTIC_LATERAL_TEST_RECORDS,
                'drop_first_last_detected_beats': PTB_DIAGNOSTIC_DROP_BOUNDARY_BEATS,
            } if dataset_name == 'ptb_diagnostic' else None,
            'split_policy': split_policy,
            'excluded_policy': 'PTB-XL keeps exact strict classes. PTB Diagnostic keeps strict normal/anterior/inferior and maps all lateral-involved localization records to Lateral, including antero-lateral, infero-lateral, postero-lateral, and infero-postero-lateral.',
        }, f, indent=2)

    print(f'\n{dataset_name} split summary')
    display(split_summary)
    return split_summary


## 6. Run Export

In [ ]:
all_summaries = []

ptbxl_labels, ptbxl_signals, ptbxl_skipped = load_ptbxl_records()
print('PTB-XL strict records:', len(ptbxl_labels), 'skipped:', len(ptbxl_skipped))
all_summaries.append(export_dataset('ptb_xl', ptbxl_labels, ptbxl_signals, ptbxl_skipped))

ptb_diag_labels, ptb_diag_signals, ptb_diag_skipped = load_ptb_diagnostic_records()
print('PTB Diagnostic strict records:', len(ptb_diag_labels), 'skipped:', len(ptb_diag_skipped))
all_summaries.append(export_dataset('ptb_diagnostic', ptb_diag_labels, ptb_diag_signals, ptb_diag_skipped))

combined_summary = pd.concat(all_summaries, ignore_index=True)
combined_summary.to_csv(OUTPUT_ROOT / 'combined_split_summary.csv', index=False)
print('Final beat-signal datasets exported successfully.')
display(combined_summary)


## 7. Validation

In [ ]:
for dataset_name in DATASETS:
    print(f'\n=== {dataset_name} ===')
    summary = pd.read_csv(OUTPUT_ROOT / dataset_name / 'split_summary.csv')
    display(summary)
    record_sets = {}
    for split_name in SPLITS:
        split_dir = OUTPUT_ROOT / dataset_name / split_name
        x = np.load(split_dir / 'x_beats.npy', mmap_mode='r')
        main = np.load(split_dir / 'main_label.npy')
        territory = np.load(split_dir / 'territory.npy')
        prior = np.load(split_dir / 'lead_prior.npy')
        meta = pd.read_csv(split_dir / 'metadata.csv')
        assert x.shape[0] == main.shape[0] == territory.shape[0] == prior.shape[0] == len(meta)
        assert x.shape[1:] == (BEAT_LENGTH, len(ALPA_LEAD_ORDER))
        assert main.shape[1] == 4 and territory.shape[1] == 3 and prior.shape[1] == 12
        assert np.allclose(main.sum(axis=1), 1.0)
        assert np.allclose(prior.sum(axis=1), 1.0)
        record_sets[split_name] = set(meta['record_id'].astype(str))
        print(f'{split_name}: x={x.shape}, labels={main.sum(axis=0).astype(int).tolist()}')
    if dataset_name == 'ptb_diagnostic':
        assert not (record_sets['train'] & record_sets['test'])
        assert not (record_sets['val'] & record_sets['test'])
        assert not (record_sets['train'] & record_sets['val'])
        split_meta = {split_name: pd.read_csv(OUTPUT_ROOT / dataset_name / split_name / 'metadata.csv') for split_name in SPLITS}
        all_meta = pd.concat([df for df in split_meta.values() if len(df)], ignore_index=True)
        fixed_mask = (
            all_meta['record_id'].astype(str).eq(PTB_DIAGNOSTIC_FIXED_LATERAL_TEST_RECORD_ID)
            & all_meta['patient_id'].astype(str).eq(PTB_DIAGNOSTIC_FIXED_LATERAL_TEST_PATIENT_ID)
            & all_meta['recordwise_split'].astype(str).eq('test')
        )
        assert fixed_mask.any(), 'Fixed lateral test record is missing from PTB Diagnostic test split.'
        assert (all_meta.loc[fixed_mask, 'source_signal_seconds'].round(6) == PTB_DIAGNOSTIC_LATERAL_SECONDS).all()
        assert (all_meta.loc[~fixed_mask, 'source_signal_seconds'].round(6) == PTB_DIAGNOSTIC_STANDARD_SECONDS).all()
        print('ptb_diagnostic record-wise checks passed; fixed lateral test record uses 60 seconds and all other records use 10 seconds.')
    else:
        for i, left in enumerate(SPLITS):
            for right in SPLITS[i + 1:]:
                assert not (record_sets[left] & record_sets[right])
print('Validation finished successfully.')
